# MERA AI CINEMATIC STUDIO — FREE COLAB EDITION

**One prompt → automatic story/shot plan → local English narration → generated video clips → synthetic score/SFX → subtitles → MP4.**

This notebook is designed for someone who only edits the settings cell and presses **Runtime → Run all**. It uses no paid API key or paid inference service. Internet is needed to download model weights.

## Before you start
1. Open this notebook in Google Colab.
2. **Before running all cells, select a GPU.** Try **Runtime → Change runtime type → Hardware accelerator → T4 GPU**. In the newer interface, use the small arrow beside **Connect** and choose **Change runtime type**. GPU access is not guaranteed on Colab Free.
3. Run **Runtime → Run all** and approve Google Drive access.
4. In the **USER SETTINGS** cell, edit `PROMPT` and `TARGET_DURATION_SECONDS`. Do not change anything else at first.
5. Final files are saved to `MyDrive/MERA_AI_CINEMATIC_STUDIO/` after every scene. If Colab disconnects, reopen this notebook and run all cells again; completed scenes are skipped where possible.

## Honest quality / hardware limits
This is a real working prototype, not a promise of Highsfield-level quality. A free Colab GPU may generate short clips slowly and can disconnect. The notebook renders a film scene-by-scene and resumes from saved clips, but a 5-minute film can take many hours or multiple sessions. The low-memory LTX-Video 2B model is a practical compromise; it will not match the newest 13B/large commercial systems in every shot. Character identity is reinforced by a consistent character bible in each prompt; an optional reference image is passed to the LTX conditioning helper when supported, but no open model guarantees perfect identity across all shots.

The score and effects are generated procedurally (synthesized tones, ambience and transition hits), not from an orchestral music model; they are original but intentionally simple. Scaling the finished clip to 720p is conventional upscaling, **not** AI super-resolution. Motion interpolation is optional and can create artifacts. The default narration is English.

**Licensing:** code being free/open does not automatically grant commercial rights to every model checkpoint. Review the model-weight terms before monetized/client work. MusicGen is intentionally not used because its official card says the weights are CC-BY-NC 4.0. Colab itself has variable, non-guaranteed resources.


In [ ]:
# ======================== USER SETTINGS ========================
# In normal use, change ONLY PROMPT and TARGET_DURATION_SECONDS.

PROMPT = "Create a 30-second cinematic 3D fantasy film: a brave hero makes a dramatic entrance into an ancient storm-lit kingdom, with a mysterious atmosphere, emotional narration, sweeping camera movement and a powerful final reveal."
TARGET_DURATION_SECONDS = 30       # e.g. 30, 60, or 300 for a 5-minute documentary
SCENE_SECONDS = 5                  # shorter scenes = more clips and more rendering time
VIDEO_STYLE = "cinematic, high-end stylized 3D animation, detailed environment, dramatic but tasteful lighting"
AUDIENCE = "general English-speaking audience"
ASPECT_RATIO = "16:9"              # use "9:16" for vertical Shorts/Reels
VOICE_LANGUAGE = "a"               # "a" = American English; "b" = British English
VOICE_NAME = "af_heart"            # US; British example: "bf_emma"
REFERENCE_IMAGE_PATH = ""           # optional: Drive path to a character/reference image; leave blank if unused
SEED = 2026

# Advanced switches — leave as they are on first run.
ENABLE_OPTIONAL_MOTION_INTERPOLATION = False
OUTPUT_FPS = 24
FINAL_LONG_EDGE = 1280              # exported video is 720p landscape or 1280px tall portrait
NEGATIVE_PROMPT = "text, subtitles, watermark, logos, duplicated people, extra limbs, deformed hands, distorted face, face morphing, flicker, jitter, sudden outfit changes, low detail, blurry, frozen motion, plastic skin, oversaturated colors"

if TARGET_DURATION_SECONDS <= 0 or SCENE_SECONDS <= 0:
    raise ValueError("TARGET_DURATION_SECONDS aur SCENE_SECONDS zero se baray hone chahiye.")
if ASPECT_RATIO not in {"16:9", "9:16"}:
    raise ValueError('ASPECT_RATIO "16:9" ya "9:16" hona chahiye.')
print("Settings loaded. Target duration:", TARGET_DURATION_SECONDS, "seconds")


In [ ]:
# EARLY GPU CHECK — runs BEFORE model downloads and package installation.
import subprocess, sys
gpu_check = subprocess.run(
    ['bash', '-lc', 'nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'],
    capture_output=True, text=True
)
if gpu_check.returncode != 0 or not gpu_check.stdout.strip():
    raise RuntimeError(
        'NO NVIDIA GPU IS CONNECTED. Stop here; do not run all cells repeatedly. '
        'In Google Colab, open Runtime > Change runtime type > Hardware accelerator > T4 GPU. '
        'In the newer UI, try the arrow beside Connect > Change runtime type. '
        'If Hardware accelerator/GPU is not offered, Colab may not currently be giving this account a free GPU. '
        'The notebook cannot enable a GPU by itself; try again later or use a runtime that already has a GPU.'
    )
try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('nvidia-smi found a GPU, but PyTorch CUDA is unavailable in this runtime. Disconnect/reconnect the runtime and try again.')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU connected: {gpu_name} | VRAM: {gpu_vram_gb:.1f} GiB')
    if gpu_vram_gb < 10:
        raise RuntimeError(
            f'This notebook needs about 10 GiB of GPU VRAM; detected {gpu_vram_gb:.1f} GiB. '
            'Choose a larger available NVIDIA GPU before continuing.'
        )
    print('GPU preflight passed. It is now safe to continue with Runtime > Run all.')
except ImportError:
    raise RuntimeError('PyTorch is not available in this runtime. Select a GPU runtime and reconnect before running the notebook.')


## 1 — Connect Google Drive and prepare folders
The GPU report is a live measurement from this runtime. The code does not assume a specific Colab GPU. The generated story, voice, each finished clip, and final MP4 will be saved in Drive.


In [ ]:
import os, sys, json, re, math, gc, time, hashlib, shutil, subprocess, textwrap, inspect, random
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Google Drive mount issue:", e)
    print("This notebook is intended for Google Colab. Open it in Colab and approve Drive access.")

DRIVE_ROOT = Path('/content/drive/MyDrive/MERA_AI_CINEMATIC_STUDIO')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

def slugify(text, max_len=36):
    text = re.sub(r'[^A-Za-z0-9]+', '-', text).strip('-').lower()
    return (text[:max_len].strip('-') or 'cinematic-project')

project_key = hashlib.sha256((PROMPT + str(TARGET_DURATION_SECONDS) + str(SCENE_SECONDS) + VIDEO_STYLE + ASPECT_RATIO + VOICE_LANGUAGE + VOICE_NAME + str(SEED) + REFERENCE_IMAGE_PATH).encode('utf-8')).hexdigest()[:10]
PROJECT_DIR = DRIVE_ROOT / f"{slugify(PROMPT)}_{project_key}"
CLIPS_DIR = PROJECT_DIR / 'clips'
AUDIO_DIR = PROJECT_DIR / 'audio'
TEMP_DIR = PROJECT_DIR / 'temp'
for p in (PROJECT_DIR, CLIPS_DIR, AUDIO_DIR, TEMP_DIR):
    p.mkdir(parents=True, exist_ok=True)

print('Project folder:', PROJECT_DIR)
print('GPU report (nvidia-smi):')
try:
    subprocess.run(['nvidia-smi'], check=False)
except Exception as e:
    print('nvidia-smi unavailable:', e)
print('System RAM (free -h):')
subprocess.run(['bash','-lc','free -h'], check=False)


## 2 — Install the free local models and video engine
This cell installs the official LTX-Video inference repo, Kokoro TTS, FFmpeg and speech dependencies. On the first run this can take a while. It does not ask for an API key or payment.


In [ ]:
# Install system dependencies
subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y -qq ffmpeg espeak-ng'], check=True)

# Get the official LTX-Video repository
REPO_DIR = Path('/content/LTX-Video')
if not (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Lightricks/LTX-Video.git', str(REPO_DIR)], check=True)

# Install official inference extras. Pin transformers inside the compatibility range used by this codebase.
subprocess.run(['bash','-lc', f"cd {REPO_DIR} && pip install -q -e '.[inference]'"], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.51.3', 'accelerate', 'sentencepiece', 'soundfile', 'kokoro>=0.9.4', 'imageio[ffmpeg]'], check=True)

# Small compatibility patch: use FP16 for T4 GPUs that don't support BF16.
# It changes only the precision selection for the model, VAE and text encoder.
inference_py = REPO_DIR / 'ltx_video' / 'inference.py'
source = inference_py.read_text(encoding='utf-8')
if 'elif precision == "float16":' not in source:
    needle = '    elif precision == "bfloat16":'
    if needle not in source:
        raise RuntimeError('LTX inference.py changed upstream: FP16 patch point not found. No model has been generated; inspect the official repo before proceeding.')
    source = source.replace(needle, '    elif precision == "float16":\n        return Transformer3DModel.from_pretrained(ckpt_path).to(torch.float16)\n' + needle, 1)
old_dtype = '    vae = vae.to(torch.bfloat16)\n    text_encoder = text_encoder.to(torch.bfloat16)'
new_dtype = '    pipeline_dtype = torch.float16 if precision == "float16" else torch.bfloat16\n    vae = vae.to(pipeline_dtype)\n    text_encoder = text_encoder.to(pipeline_dtype)'
if old_dtype in source:
    source = source.replace(old_dtype, new_dtype, 1)
elif new_dtype not in source:
    raise RuntimeError('Could not apply safe FP16 patch to VAE/text encoder. The upstream code changed; stop here rather than risk a broken setup.')
inference_py.write_text(source, encoding='utf-8')

print('Install complete. LTX inference code patched for FP16 compatibility.')
print('If pip reports dependency conflicts, restart the runtime once and run this notebook again from the top.')


## 3 — Detect GPU and choose settings before rendering
The renderer selects resolution and frame count from actual VRAM. A GPU below the safety floor is stopped before loading the video model, rather than pretending CPU generation will be practical.


In [ ]:
import torch
import psutil

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU: {GPU_NAME} | VRAM: {GPU_VRAM_GB:.1f} GiB')
else:
    GPU_NAME, GPU_VRAM_GB = 'No CUDA GPU', 0.0
    print('No CUDA GPU detected. You can still inspect the plan, but AI video generation needs a CUDA GPU.')
print(f'System RAM: {psutil.virtual_memory().total / (1024**3):.1f} GiB')

# Resolution choices are aligned to 32-pixel dimensions for the LTX video VAE.
if GPU_VRAM_GB >= 22:
    BASE_W, BASE_H, BASE_FRAMES = (640, 352, 65)
elif GPU_VRAM_GB >= 14:
    BASE_W, BASE_H, BASE_FRAMES = (512, 288, 49)
elif GPU_VRAM_GB >= 10:
    BASE_W, BASE_H, BASE_FRAMES = (384, 224, 33)
else:
    BASE_W, BASE_H, BASE_FRAMES = (0, 0, 0)

if ASPECT_RATIO == '9:16' and BASE_W:
    BASE_W, BASE_H = BASE_H, BASE_W

print('Automatic render profile:', {'width': BASE_W, 'height': BASE_H, 'frames': BASE_FRAMES, 'fps': OUTPUT_FPS})
if GPU_VRAM_GB < 10:
    print('SAFETY STOP: this GPU is below the notebook safety floor (~10 GiB). The renderer will not try a likely-OOM configuration.')


## 4 — AI Director: story, consistent character bible and shot list
A small local instruction model (`Qwen2.5-1.5B-Instruct`) drafts the shot list on CPU, so it does not compete with the video model for VRAM. Every shot repeats a shared visual/character bible. If the local director output is malformed, the notebook falls back to a valid template rather than stopping the whole project.


In [ ]:
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

DIRECTOR_MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
PLAN_PATH = PROJECT_DIR / 'production_plan.json'

n_scenes = int(math.ceil(TARGET_DURATION_SECONDS / SCENE_SECONDS))
slot_seconds = [min(float(SCENE_SECONDS), TARGET_DURATION_SECONDS - i * SCENE_SECONDS) for i in range(n_scenes)]

# Reuse a saved plan unless the user explicitly asks for a rebuild.
if PLAN_PATH.exists():
    plan = json.loads(PLAN_PATH.read_text(encoding='utf-8'))
    if plan.get('target_duration') != TARGET_DURATION_SECONDS or plan.get('prompt') != PROMPT:
        print('Saved plan does not match settings; making a fresh plan.')
        plan = None
else:
    plan = None

if plan is None:
    print('Loading local AI Director (first run downloads the model)…')
    director_tokenizer = AutoTokenizer.from_pretrained(DIRECTOR_MODEL_ID)
    director_model = AutoModelForCausalLM.from_pretrained(DIRECTOR_MODEL_ID, torch_dtype=torch.float32, low_cpu_mem_usage=True)
    director_model.to('cpu').eval()

    def ask_director(user_text, max_new_tokens=500):
        messages = [
            {'role':'system', 'content':'You are a concise, professional film director and documentary writer. Write clear natural English. Follow the requested format exactly. No markdown unless explicitly requested.'},
            {'role':'user', 'content':user_text}
        ]
        rendered = director_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = director_tokenizer(rendered, return_tensors='pt')
        with torch.inference_mode():
            out = director_model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=True,
                temperature=0.65, top_p=0.9, repetition_penalty=1.08,
                pad_token_id=director_tokenizer.eos_token_id
            )
        return director_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    bible_request = f"""Create a compact visual continuity bible for this film idea. PROMPT: {PROMPT}\nSTYLE: {VIDEO_STYLE}\nAudience: {AUDIENCE}\nIn 60-90 words, define the main subject/character appearance, age-range if relevant, face/hair/clothing/material colours, body proportions, key props, environment, colour palette and lighting. If no human character is needed, define the recurring subject and environment instead. Keep it easy to reuse in every shot. Return only the bible, no title."""
    try:
        character_bible = ask_director(bible_request, max_new_tokens=180)
    except Exception as e:
        print('Director bible fallback:', e)
        character_bible = f"Continuity bible: keep the main subject from the brief visually identical across every shot; preserve silhouette, face, wardrobe, colours, props, location, and cinematic lighting. Style: {VIDEO_STYLE}."

    scenes = []
    batch_size = 5
    for bstart in range(0, n_scenes, batch_size):
        ids = list(range(bstart, min(n_scenes, bstart + batch_size)))
        slot_info = '\n'.join([f"Shot {i+1}: target length {slot_seconds[i]:.1f} seconds; narration should be about {max(3, round(slot_seconds[i]*2.7))} words." for i in ids])
        req = f"""Develop the next part of a coherent cinematic film. Return ONLY a valid JSON array with exactly {len(ids)} objects. Each object must have these string keys: narration, visual_prompt, camera, sound_design. No markdown fences, no extra keys. Each narration is simple, natural English voice-over, roughly the word count specified below, no numbering, no quotation marks. visual_prompt must describe a specific visible action/environment, not abstract ideas. Keep the plot continuous and the same subject identity through all shots. Avoid written text in the image.\nFILM BRIEF: {PROMPT}\nSTYLE: {VIDEO_STYLE}\nAUDIENCE: {AUDIENCE}\nVISUAL/CHARACTER BIBLE (do not contradict it): {character_bible}\nSHOT REQUIREMENTS:\n{slot_info}\nSequence position: shots {bstart+1} to {ids[-1]+1} of {n_scenes}. Use varied but motivated wide/medium/close-up framing and smooth physical camera movement. Sound design should be a short cue such as low impact, wind, soft whoosh, ambience. The narration should move the story forward rather than repeat the prompt."""
        raw = ''
        try:
            raw = ask_director(req, max_new_tokens=min(1800, max(400, len(ids)*220)))
            raw_clean = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw.strip(), flags=re.I)
            match = re.search(r'\[[\s\S]*\]', raw_clean)
            batch = json.loads(match.group(0)) if match else None
            if not isinstance(batch, list) or len(batch) != len(ids):
                raise ValueError('JSON array missing or wrong item count')
        except Exception as e:
            print(f'Shot batch {bstart+1}-{ids[-1]+1} used fallback prompts ({e}).')
            batch = []
            for i in ids:
                batch.append({
                    'narration': f"{PROMPT[:80].rstrip('.')} unfolds in a vivid cinematic sequence, revealing a new detail and building anticipation."[:500],
                    'visual_prompt': f"A coherent shot illustrating the next story beat of: {PROMPT}. Sequence shot {i+1} of {n_scenes}; clear subject action, rich environment, cinematic depth, physically plausible motion.",
                    'camera': 'slow push-in with a gentle lateral drift, stable horizon',
                    'sound_design': 'soft atmospheric whoosh and low cinematic impact'
                })
        for j, item in zip(ids, batch):
            if not isinstance(item, dict): item = {}
            narration = re.sub(r'\s+', ' ', str(item.get('narration', '')).strip())
            if not narration:
                narration = f"The story moves forward, revealing another detail of {PROMPT[:70]} and raising the stakes."
            scenes.append({
                'scene_id': j + 1,
                'duration_seconds': slot_seconds[j],
                'narration': narration,
                'visual_prompt': str(item.get('visual_prompt') or f"Cinematic visual of {PROMPT}, shot {j+1} of {n_scenes}."),
                'camera': str(item.get('camera') or 'slow cinematic dolly movement'),
                'sound_design': str(item.get('sound_design') or 'low atmospheric whoosh')
            })
        print(f'Planned {min(bstart+batch_size,n_scenes)}/{n_scenes} shots')

    plan = {
        'project_title': slugify(PROMPT),
        'prompt': PROMPT,
        'target_duration': TARGET_DURATION_SECONDS,
        'scene_seconds': SCENE_SECONDS,
        'video_style': VIDEO_STYLE,
        'aspect_ratio': ASPECT_RATIO,
        'character_bible': character_bible,
        'scenes': scenes,
        'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
    }
    PLAN_PATH.write_text(json.dumps(plan, ensure_ascii=False, indent=2), encoding='utf-8')
    del director_model, director_tokenizer
    gc.collect()

print('\nCHARACTER / VISUAL BIBLE:\n', plan.get('character_bible',''))
print(f"\nProduction plan: {len(plan['scenes'])} scenes")
for s in plan['scenes'][:3]:
    print(f"\nSHOT {s['scene_id']} ({s['duration_seconds']}s)\nVO: {s['narration']}\nVISUAL: {s['visual_prompt']}\nCAMERA: {s['camera']}")
print('\nFull plan saved to:', PLAN_PATH)


## 5 — English narration and original synthesized score
Kokoro-82M generates local English speech. The background score and transition hits are synthesized using signal processing, so no external music API or non-commercial-only music model is called. Voice lines are gently time-adjusted to fit the chosen scene timeline where possible.


In [ ]:
import soundfile as sf
from kokoro import KPipeline

voice_files = []
print('Loading Kokoro TTS. It runs locally; first run downloads its weights.')
tts = KPipeline(lang_code=VOICE_LANGUAGE)

for idx, scene in enumerate(plan['scenes'], start=1):
    wav_path = AUDIO_DIR / f'narration_{idx:04d}.wav'
    fixed_path = AUDIO_DIR / f'narration_fixed_{idx:04d}.wav'
    voice_files.append(fixed_path)
    if fixed_path.exists():
        continue
    if not wav_path.exists():
        chunks = []
        for _, _, audio_chunk in tts(scene['narration'], voice=VOICE_NAME):
            if audio_chunk is not None and len(audio_chunk):
                chunks.append(np.asarray(audio_chunk, dtype=np.float32).reshape(-1))
        if not chunks:
            raise RuntimeError(f'Kokoro did not return speech for scene {idx}. Check VOICE_LANGUAGE/VOICE_NAME.')
        wav = np.concatenate(chunks)
        peak = float(np.max(np.abs(wav))) if wav.size else 0.0
        if peak > 0.98: wav = wav * (0.95 / peak)
        sf.write(str(wav_path), wav, 24000)

    audio, sr = sf.read(str(wav_path), dtype='float32')
    if audio.ndim > 1: audio = audio.mean(axis=1)
    actual = len(audio) / sr
    target = float(scene['duration_seconds'])
    speed = actual / target if target > 0 else 1.0
    # ffmpeg atempo keeps the voice waveform/pitch more natural than raw resampling.
    if 0.50 <= speed <= 2.00 and abs(actual - target) > 0.12:
        subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(wav_path),'-filter:a',f'atempo={speed:.6f}','-ar','24000','-ac','1',str(fixed_path)], check=True)
    elif abs(actual - target) <= 0.12:
        shutil.copy2(wav_path, fixed_path)
    else:
        # Keep the natural speech length if extreme stretching would sound bad.
        shutil.copy2(wav_path, fixed_path)
        print(f'Note: narration {idx} is {actual:.1f}s for a {target:.1f}s slot; final scene may run long.')
    print(f'Voice {idx}/{len(plan["scenes"])} prepared')

del tts
gc.collect()

# Inspect fixed audio duration per scene and create a merged narration file.
scene_audio_durations = []
scene_audio_arrays = []
for path in voice_files:
    aud, sr = sf.read(str(path), dtype='float32')
    if aud.ndim > 1: aud = aud.mean(axis=1)
    scene_audio_arrays.append(aud)
    scene_audio_durations.append(len(aud) / sr)

voice_concat = np.concatenate(scene_audio_arrays) if scene_audio_arrays else np.zeros(1, dtype=np.float32)
full_voice_path = AUDIO_DIR / 'full_voiceover.wav'
sf.write(str(full_voice_path), voice_concat, 24000)
voice_duration = len(voice_concat) / 24000
print(f'Voice-over ready: {voice_duration:.1f}s | {full_voice_path}')


In [ ]:
# Make an original, simple cinematic synth bed + transition hits (not a sampled song).
# Chord pads + low moving drone + softened impacts at scene boundaries.
def make_procedural_score(duration, scene_lengths, out_path, sr=24000, seed=2026):
    rng = np.random.default_rng(seed)
    n = max(1, int(duration * sr))
    t = np.arange(n, dtype=np.float32) / sr
    score = np.zeros(n, dtype=np.float32)
    # Calm minor-key progression, 8-second cycles. Each chord is a set of frequencies.
    chords = [
        [65.41, 98.00, 130.81, 155.56],  # C minor-ish
        [51.91, 77.78, 103.83, 130.81],  # Ab
        [58.27, 87.31, 116.54, 146.83],  # Bb
        [49.00, 73.42, 98.00, 116.54],   # G minor-ish
    ]
    chord_seconds = 8.0
    for start in np.arange(0, duration, chord_seconds):
        end = min(duration, start + chord_seconds)
        i0, i1 = int(start * sr), min(n, int(end * sr))
        tt = t[i0:i1] - start
        if len(tt) == 0: continue
        env = np.minimum(1.0, tt / 1.5) * np.minimum(1.0, (end - start - tt) / 1.4)
        pad = np.zeros_like(tt)
        chord = chords[int(start // chord_seconds) % len(chords)]
        for j, f in enumerate(chord):
            pad += (0.45 / (j + 1)) * np.sin(2 * np.pi * f * tt + j * 0.37)
            pad += (0.12 / (j + 1)) * np.sin(2 * np.pi * (f * 2.003) * tt + j * 0.17)
        trem = 0.85 + 0.15 * np.sin(2 * np.pi * 0.09 * tt)
        score[i0:i1] += 0.12 * pad * env * trem
    # Quiet sub-bass drone with slow movement
    drone = np.sin(2*np.pi*(42 + 2*np.sin(2*np.pi*0.025*t))*t)
    score += 0.045 * drone
    # Gentle transition impact + noise-whoosh at every shot boundary.
    cursor = 0.0
    for k, dur in enumerate(scene_lengths):
        if k > 0:
            idx = int(cursor * sr)
            if idx < n:
                length = min(int(1.1*sr), n-idx)
                tt = np.arange(length, dtype=np.float32) / sr
                decay = np.exp(-tt * (4.2 if k % 2 else 3.0))
                hit = 0.20*np.sin(2*np.pi*(58-18*np.minimum(tt,0.7))*tt)*decay
                noise = rng.normal(0, 1, length).astype(np.float32)
                # Short noisy swoosh, intentionally quiet under narration.
                swoosh_env = np.sin(np.pi*np.minimum(tt/0.7, 1.0))**2 * np.exp(-tt*2.0)
                hit += 0.025*noise*swoosh_env
                score[idx:idx+length] += hit
        cursor += dur
    # Fade in/out and stereo spread.
    fade_n = min(int(1.4*sr), n//3)
    if fade_n:
        score[:fade_n] *= np.linspace(0, 1, fade_n, dtype=np.float32)
        score[-fade_n:] *= np.linspace(1, 0, fade_n, dtype=np.float32)
    peak = np.max(np.abs(score)) or 1.0
    score = score / max(1.0, peak/0.65)
    left = score
    right = np.roll(score, int(0.006*sr)) * 0.97
    stereo = np.stack([left, right], axis=1).astype(np.float32)
    sf.write(str(out_path), stereo, sr)
    return duration

score_path = AUDIO_DIR / 'original_synth_score.wav'
make_procedural_score(voice_duration, scene_audio_durations, score_path, seed=SEED)
print('Synth score ready:', score_path)


## 6 — Generate each video shot (automatically resumes)
The official LTX-Video 2B distilled checkpoint is the lightweight choice here. The notebook picks dimensions from the detected GPU, uses 8 inference steps, saves each scene to Drive, and retries one failed out-of-memory scene at a smaller size. A reference image is optional; its use depends on the version of the official conditioning helper installed with LTX-Video.


In [ ]:
if GPU_VRAM_GB < 10:
    raise RuntimeError('Video generation stopped before model download: detected <10 GiB VRAM. Reconnect with a CUDA GPU that has more VRAM; plan/audio files are already saved.')

from huggingface_hub import hf_hub_download
from ltx_video.utils.skip_layer_strategy import SkipLayerStrategy
import imageio.v2 as imageio

# Helpers from the official LTX inference module
os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path: sys.path.insert(0, str(REPO_DIR))
import inference as ltx_inference

VIDEO_MODEL_REPO = 'Lightricks/LTX-Video'
VIDEO_CHECKPOINT = 'ltxv-2b-0.9.6-distilled-04-25.safetensors'
TEXT_ENCODER = 'PixArt-alpha/PixArt-XL-2-1024-MS'
print('Downloading/locating official LTX-Video 2B distilled checkpoint…')
ckpt_path = hf_hub_download(repo_id=VIDEO_MODEL_REPO, filename=VIDEO_CHECKPOINT)

builder = ltx_inference.create_ltx_video_pipeline
build_sig = inspect.signature(builder)
build_args = {
    'ckpt_path': ckpt_path,
    'precision': 'float16',
    'text_encoder_model_name_or_path': TEXT_ENCODER,
    'sampler': 'from_checkpoint',
    'device': 'cuda',
    'enhance_prompt': False,
}
pipe = builder(**{k:v for k,v in build_args.items() if k in build_sig.parameters})
print('Video pipeline loaded with FP16.')

# Prepare optional reference image conditioning. Failure is non-fatal; plan still uses the character bible.
def get_reference_conditioning(path, height, width, num_frames):
    if not path or not Path(path).exists():
        return None
    helper = getattr(ltx_inference, 'prepare_conditioning', None)
    if helper is None:
        print('This LTX repo version has no prepare_conditioning helper; continuing without reference conditioning.')
        return None
    try:
        sig = inspect.signature(helper)
        offered = {
            'conditioning_media_paths': [str(path)],
            'media_paths': [str(path)],
            'conditioning_start_frames': [0],
            'start_frames': [0],
            'conditioning_strengths': [0.8],
            'strengths': [0.8],
            'height': height, 'width': width, 'num_frames': num_frames,
            'vae': getattr(pipe, 'vae', None), 'device': 'cuda', 'dtype': torch.float16,
            'precision': 'float16', 'just_crop': False
        }
        kwargs = {k:v for k,v in offered.items() if k in sig.parameters}
        result = helper(**kwargs)
        if isinstance(result, tuple):
            result = result[0]
        return result
    except Exception as e:
        print('Reference conditioning could not be initialized; using character-bible text conditioning instead. Detail:', str(e)[:250])
        return None

# Render each missing shot. Every shot includes the same identity bible and the same negative prompt.
render_log = PROJECT_DIR / 'render_log.txt'
with render_log.open('a', encoding='utf-8') as log:
    for idx, scene in enumerate(plan['scenes'], start=1):
        clip_path = CLIPS_DIR / f'scene_{idx:04d}.mp4'
        if clip_path.exists() and clip_path.stat().st_size > 10_000:
            print(f'[{idx}/{len(plan["scenes"])}] already exists — skipping')
            continue
        visual_prompt = (
            f"{scene['visual_prompt']}\n"
            f"Visual continuity bible: {plan.get('character_bible','')}\n"
            f"Camera: {scene['camera']}.\n"
            f"Overall style: {VIDEO_STYLE}. Natural physical movement, coherent anatomy, consistent wardrobe and subject identity, cinematic depth, realistic light interaction. No text or logo."
        )
        h, w, frames = BASE_H, BASE_W, BASE_FRAMES
        conditioning = get_reference_conditioning(REFERENCE_IMAGE_PATH, h, w, frames)
        attempts = [(h, w, frames, conditioning)]
        # A genuinely smaller retry profile for each VRAM tier.
        if GPU_VRAM_GB >= 22:
            retry_w, retry_h = ((288, 512) if ASPECT_RATIO == '9:16' else (512, 288))
        elif GPU_VRAM_GB >= 14:
            retry_w, retry_h = ((256, 448) if ASPECT_RATIO == '9:16' else (448, 256))
        else:
            retry_w, retry_h = ((192, 320) if ASPECT_RATIO == '9:16' else (320, 192))
        attempts.append((retry_h, retry_w, 33, None))
        succeeded = False
        last_error = None
        for attempt_idx, (ah, aw, af, acond) in enumerate(attempts, start=1):
            try:
                run_kwargs = {
                    'prompt': visual_prompt,
                    'prompt_attention_mask': None,
                    'negative_prompt': NEGATIVE_PROMPT,
                    'negative_prompt_attention_mask': None,
                    'height': int(ah), 'width': int(aw), 'num_frames': int(af),
                    'frame_rate': OUTPUT_FPS,
                    'num_inference_steps': 8,
                    'guidance_scale': 1.0,
                    'stg_scale': 0.0,
                    'rescaling_scale': 1.0,
                    'skip_layer_strategy': SkipLayerStrategy.AttentionValues,
                    'generator': torch.Generator(device='cuda').manual_seed(SEED + idx),
                    'decode_timestep': 0.05,
                    'decode_noise_scale': 0.025,
                    'stochastic_sampling': True,
                    'output_type': 'pt',
                    'is_video': True,
                    'vae_per_channel_normalize': True,
                    'offload_to_cpu': GPU_VRAM_GB < 24,
                    'device': 'cuda',
                    'mixed_precision': False,
                    'image_cond_noise_scale': 0.15,
                }
                if acond is not None:
                    run_kwargs['conditioning_items'] = acond
                pipe_sig = inspect.signature(pipe.__call__)
                # Keep only parameters actually supported by this installed official version.
                if not any(p.kind == inspect.Parameter.VAR_KEYWORD for p in pipe_sig.parameters.values()):
                    run_kwargs = {k:v for k,v in run_kwargs.items() if k in pipe_sig.parameters}
                with torch.inference_mode():
                    result = pipe(**run_kwargs)
                frames_out = result.images if hasattr(result, 'images') else result[0]
                if isinstance(frames_out, np.ndarray):
                    arr = frames_out
                    if arr.ndim == 5: arr = arr[0]
                    if arr.shape[0] in (1,3,4): arr = np.transpose(arr, (1,2,3,0))
                    if arr.dtype != np.uint8:
                        if arr.min() < 0: arr = (arr + 1.0) / 2.0
                        arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
                else:
                    tensor = frames_out[0] if frames_out.ndim == 5 else frames_out
                    # Expected shape [C, F, H, W] -> [F, H, W, C]
                    arr = tensor.detach().float().cpu().permute(1,2,3,0).numpy()
                    if arr.min() < 0: arr = (arr + 1.0) / 2.0
                    arr = np.clip(arr * 255.0, 0, 255).astype(np.uint8)
                if arr.ndim != 4 or arr.shape[-1] < 3:
                    raise ValueError(f'Unexpected generated frame shape {arr.shape}')
                arr = arr[..., :3]
                tmp_clip = TEMP_DIR / f'scene_{idx:04d}_new.mp4'
                imageio.mimwrite(str(tmp_clip), arr, fps=OUTPUT_FPS, codec='libx264', quality=8, macro_block_size=None)
                shutil.move(str(tmp_clip), str(clip_path))
                msg = f'OK scene {idx} ({aw}x{ah}, {af} frames, attempt {attempt_idx}) -> {clip_path}'
                print(msg); log.write(msg+'\n'); log.flush()
                succeeded = True
                del result, frames_out, arr
                gc.collect(); torch.cuda.empty_cache()
                break
            except Exception as e:
                last_error = e
                print(f'Scene {idx}, attempt {attempt_idx} failed: {str(e)[:350]}')
                log.write(f'FAIL scene {idx} attempt {attempt_idx}: {repr(e)}\n'); log.flush()
                gc.collect()
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                # Retry only for memory-related errors or the first failed attempt.
                if attempt_idx >= len(attempts): break
                if 'out of memory' not in str(e).lower() and 'cuda error' not in str(e).lower():
                    # One retry can also handle occasional transient CUDA/model errors.
                    pass
        if not succeeded:
            raise RuntimeError(f'Scene {idx} could not be generated. Earlier scenes remain safely saved in Drive. Error: {last_error}')

print('All scene clips exist. Clips folder:', CLIPS_DIR)


## 7 — Assemble, subtitle, mix audio and export MP4
The assembly uses clean cuts (a valid cinematic transition), cloned-frame padding/retiming to each narration slot, a procedural score under the speech, embedded SRT captions and H.264 MP4 export. If you enable interpolation it is attempted on final export, but may be slower and can add artifacts.


In [ ]:
# First prepare a video-only segment of the requested duration for each generated clip.
prepared_segments = []
actual_voice_lengths = []
for idx, scene in enumerate(plan['scenes'], start=1):
    source_clip = CLIPS_DIR / f'scene_{idx:04d}.mp4'
    voice_path = AUDIO_DIR / f'narration_fixed_{idx:04d}.wav'
    segment_path = TEMP_DIR / f'segment_{idx:04d}.mp4'
    if not source_clip.exists():
        raise FileNotFoundError(f'Missing generated scene: {source_clip}')
    audio, sr = sf.read(str(voice_path), dtype='float32')
    if audio.ndim > 1: audio = audio.mean(axis=1)
    voice_len = len(audio) / sr
    actual_voice_lengths.append(voice_len)
    # If voice was gently retimed, use its actual duration; otherwise preserve natural speech and let final timeline expand.
    target_len = max(float(scene['duration_seconds']), voice_len)
    probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(source_clip)], capture_output=True, text=True, check=True)
    source_len = max(0.1, float(probe.stdout.strip()))
    stretch = target_len / source_len
    vf_parts = [f'setpts={stretch:.8f}*PTS', 'tpad=stop_mode=clone:stop_duration=2', f'fps={OUTPUT_FPS}']
    if ASPECT_RATIO == '9:16':
        out_w, out_h = 720, 1280
    else:
        out_w, out_h = 1280, 720
    vf_parts += [f'scale={out_w}:{out_h}:force_original_aspect_ratio=decrease',
                 f'pad={out_w}:{out_h}:(ow-iw)/2:(oh-ih)/2', 'setsar=1', 'format=yuv420p']
    # -t sets segment duration; all segments use common frame rate/resolution/codec for reliable concatenation.
    subprocess.run([
        'ffmpeg','-y','-loglevel','error','-i',str(source_clip),
        '-vf',','.join(vf_parts), '-t',f'{target_len:.4f}', '-an',
        '-r',str(OUTPUT_FPS), '-c:v','libx264','-preset','veryfast','-crf','19','-pix_fmt','yuv420p',
        str(segment_path)
    ], check=True)
    prepared_segments.append(segment_path)
    print(f'Prepared segment {idx}/{len(plan["scenes"])}: {target_len:.2f}s')

concat_file = TEMP_DIR / 'concat_list.txt'
concat_file.write_text('\n'.join([f"file '{p.as_posix()}'" for p in prepared_segments]) + '\n', encoding='utf-8')
silent_video = PROJECT_DIR / 'silent_timeline.mp4'
subprocess.run(['ffmpeg','-y','-loglevel','error','-f','concat','-safe','0','-i',str(concat_file),'-an','-c:v','libx264','-preset','veryfast','-crf','19','-pix_fmt','yuv420p',str(silent_video)], check=True)

# Merge voiceover and synthesized score into one audio file so output always includes narration.
score_for_mix = AUDIO_DIR / 'original_synth_score.wav'
mixed_audio = AUDIO_DIR / 'mixed_audio.wav'
subprocess.run([
    'ffmpeg','-y','-loglevel','error','-i',str(full_voice_path),'-i',str(score_for_mix),
    '-filter_complex','[0:a]volume=1.0[voice];[1:a]volume=0.20[score];[voice][score]amix=inputs=2:duration=first:dropout_transition=2,alimiter=limit=0.96[a]',
    '-map','[a]','-ar','24000','-ac','2',str(mixed_audio)
], check=True)

# SRT timing follows the narration-fixed timeline (each line belongs to its own scene).
def fmt_srt(seconds):
    ms = int(round(seconds * 1000))
    h = ms // 3_600_000; ms %= 3_600_000
    m = ms // 60_000; ms %= 60_000
    s = ms // 1000; ms %= 1000
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'

srt_path = PROJECT_DIR / 'subtitles.srt'
subs = []
cursor = 0.0
for idx, scene in enumerate(plan['scenes'], start=1):
    duration = max(float(scene['duration_seconds']), actual_voice_lengths[idx-1])
    subs += [str(idx), f'{fmt_srt(cursor)} --> {fmt_srt(cursor+duration)}', scene['narration'], '']
    cursor += duration
srt_path.write_text('\n'.join(subs), encoding='utf-8')

final_video = PROJECT_DIR / 'FINAL_CINEMATIC_FILM.mp4'
# Escape path for ffmpeg subtitles filter on Linux.
subtitle_filter_path = str(srt_path).replace('\\','\\\\').replace(':','\\:').replace("'", "\\'")
video_filters = []
if ENABLE_OPTIONAL_MOTION_INTERPOLATION:
    video_filters.append('minterpolate=fps=48:mi_mode=mci:mc_mode=aobmc:vsbmc=1')
video_filters.append(f"subtitles='{subtitle_filter_path}':force_style='FontName=DejaVu Sans,FontSize=24,Outline=2,Shadow=1,Alignment=2,MarginV=42'")
video_filters.append('format=yuv420p')

subprocess.run([
    'ffmpeg','-y','-loglevel','error','-i',str(silent_video),'-i',str(mixed_audio),
    '-vf',','.join(video_filters),
    '-map','0:v:0','-map','1:a:0','-c:v','libx264','-preset','medium','-crf','18',
    '-c:a','aac','-b:a','192k','-shortest','-movflags','+faststart',str(final_video)
], check=True)

probe = subprocess.run(['ffprobe','-v','error','-show_entries','format=duration,size','-of','json',str(final_video)], capture_output=True, text=True, check=True)
meta = json.loads(probe.stdout)['format']
print('\nDONE! Final MP4:', final_video)
print(f"Duration: {float(meta.get('duration',0)):.1f} seconds | File size: {float(meta.get('size',0))/1024/1024:.1f} MiB")
print('Subtitles:', srt_path)
print('Full narration:', full_voice_path)
print('All generated clips and the plan remain saved in Drive for resuming/re-editing.')


In [ ]:
from IPython.display import display, Video, FileLink
if final_video.exists():
    display(Video(str(final_video), embed=False, width=800))
    display(FileLink(str(final_video)))
else:
    print('Final video is not ready yet. Run the assembly cell again after all scenes are generated.')


## Troubleshooting / how to resume
- **Colab disconnects:** reopen the notebook, Run all again. The project plan/audio/finished scene clips remain in Drive; rendered scenes that already exist are skipped. Hugging Face model weights may need to download again in a fresh runtime.
- **GPU not available / below 10 GiB:** switch runtime to an available NVIDIA GPU. Do not pay or enter an API key.
- **Out of memory:** the notebook selects a profile from VRAM and retries a failed shot once at a lower spatial resolution with 33 frames. If that still fails, retry after reconnecting to a roomier GPU.
- **The same project folder is reused:** project ID derives from prompt, duration, style and aspect ratio. To force a completely fresh storyboard for the identical settings, delete that project folder from Drive and run again. The folder name changes automatically when the prompt, duration, scene length, aspect ratio, voice or seed changes.
- **Reference image:** upload the image to Drive and put its full path in `REFERENCE_IMAGE_PATH`. Conditioning availability depends on the version of the official LTX helper; if it fails, the notebook will keep the generated character bible and continue instead of crashing.
- **Commercial use:** check the specific video-checkpoint license before monetizing or delivering client work. “Free download” is not the same as unrestricted commercial permission.
